# Interactive Applications (VisualizerApp)

**Part I · Visualization** — Tutorial 15

Build a complete interactive application with `VisualizerApp` — a managed
lifecycle (`init` → block → `cleanup`), an async handler contract, and clean
shutdown. This chapter is the **overview hub** for interactivity; it maps out
the moving parts and points to the detail chapters:

- Controls → [Tutorial 16](../16_controls/)
- Banners & dialogs → [Tutorial 17](../17_banners_dialogs/)
- Responsive computation → [Tutorial 18](../18_responsive_computation/)


## Setup


In [1]:
from pytanga.geometry import Point, Sphere
from pytanga.viz import ControlEvent, VisualizerApp


## 1. The managed lifecycle

Derive from `VisualizerApp`, override `init()` and `cleanup()`, and call
`run()`. `self.viz` is a `Visualizer` instance available in every method.


In [2]:
class MyApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="My Interactive App")

    async def init(self) -> None:
        # Build the scene and register controls.
        self._sphere = self.viz(Sphere(Point(0, 0, 0), radius=1), opacity=0.3)
        self.viz.add_slider(
            "radius", label="Radius", min=0.2, max=5.0, value=1.0,
            on_change=self.on_radius,
        )
        self.viz.flush()

    async def on_radius(self, value: float, _event: ControlEvent) -> None:
        self.viz.update_entity(self._sphere.id, Sphere(Point(0, 0, 0), value))
        self.viz.flush()

    async def cleanup(self) -> None:
        pass  # teardown

# if __name__ == "__main__":
#     MyApp().run()
print("MyApp defined — run it with MyApp().run()")


MyApp defined — run it with MyApp().run()


## 2. The async handler contract

Every control/interaction handler is **async** and receives
`(value, event)` where `event` is a `ControlEvent` (currently carrying a
`browser_id`). Handlers run on the server's event loop, so they must not block
(see [Tutorial 18](../18_responsive_computation/)).


## 3. Clean shutdown

Stop the app three ways: terminal **Ctrl+C** (always), the browser **Ctrl+Q**
stop key (opt-in via `enable_server_stop_key=True`), or `request_shutdown()`
from any handler (e.g. a "Quit" button). `cleanup()` runs in every case.


In [3]:
class QuitApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="Quit App", enable_server_stop_key=True)  # Ctrl+Q in browser

    async def init(self) -> None:
        self.viz.add_button("quit", label="Quit", on_click=self.on_quit)

    async def on_quit(self, _value, _event) -> None:
        self.request_shutdown()

print("QuitApp defined")


QuitApp defined


## 4. Panel controls + view controls in a `SplitView`

Combine **panel controls** (`add_slider`, …) with **view controls**
(`SliderView`, …) inside a `SplitView` layout. `VisualizerApp` does not open
layouts by default, so override `run()` to call `show(layout=...)`.


In [4]:
import asyncio

from pytanga.viz import (
    ButtonView, CameraConfig3d, GroupView, SceneView, Size, SliderView, SplitView,
)

class SplitApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="My Split-View App")
        self._main_view = SceneView("")
        self._layout = self._build_layout()

    def _build_layout(self):
        return SplitView(
            orientation="horizontal",
            children=[
                GroupView(
                    "Controls",
                    [
                        SliderView("radius", label="Radius", min=0.2, max=5.0, value=1.0,
                                   on_change=self.on_radius),
                        ButtonView("btn_topdown", label="Top-down", on_click=self.on_topdown),
                    ],
                ),
                self._main_view,
            ],
        )

    def run(self, *, wait_for_browser=True, timeout=30.0):
        self.viz.show(layout=self._layout, wait_for_browser=wait_for_browser)
        asyncio.run(self._app_main())

    async def init(self) -> None:
        self._sphere = self.viz(Sphere(Point(0, 0, 0), radius=1.0), opacity=0.3)
        self.viz.flush()

    async def on_radius(self, value: float, _event: ControlEvent) -> None:
        self.viz.update_entity(self._sphere.id, Sphere(Point(0, 0, 0), radius=value))
        self.viz.flush()

    async def on_topdown(self, _value, _event) -> None:
        self.viz.set_view_camera(
            self._main_view,
            CameraConfig3d(position=(0.0, 8.0, 0.0), target=(0.0, 0.0, 0.0)),
        )

print("SplitApp defined")


SplitApp defined


## Visual Examples

The library's `two_spheres_interact.py` pattern — two spheres with a slider,
dropdown, and reset button — condensed into a `VisualizerApp`.


In [5]:
class TwoSpheresApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="Two Spheres")
        self.x = 2.5

    async def init(self) -> None:
        self.viz.add(Sphere(Point(0, 0, 0), 1.0), entity_id="a", color="#ff4444", opacity=0.3, label="$S_1$")
        self.viz.add(Sphere(Point(self.x, 0, 0), 1.3), entity_id="b", color="#4488ff", opacity=0.3, label="$S_2$")
        self.viz.add_slider("x", label="X Position", min=-3.5, max=3.5, step=0.02, value=self.x, on_change=self.on_x)
        self.viz.add_dropdown("mode", label="Display", options=["Both", "A only", "B only"], value="Both", on_change=self.on_mode)
        self.viz.add_button("reset", label="Reset", on_click=self.on_reset)
        self.viz.flush()

    async def on_x(self, value: float, _event: ControlEvent) -> None:
        self.x = value
        self.viz.update_entity("b", Sphere(Point(value, 0, 0), 1.3))
        self.viz.flush()

    async def on_mode(self, mode: str, _event: ControlEvent) -> None:
        self.viz.update("a", opacity=0.3 if mode in ("Both", "A only") else 0.0)
        self.viz.update("b", opacity=0.3 if mode in ("Both", "B only") else 0.0)
        self.viz.flush()

    async def on_reset(self, _value, _event: ControlEvent) -> None:
        await self.on_x(2.5, _event)

# TwoSpheresApp().run()
print("TwoSpheresApp defined")


TwoSpheresApp defined


## Summary

| Task | API |
|---|---|
| App base | `class MyApp(VisualizerApp):` |
| Lifecycle | `init()` → block → `cleanup()` |
| Build scene | `self.viz.add(...)` / `self.viz(...)` in `init()` |
| Handlers | `async def on_*(self, value, event)` |
| Event | `ControlEvent` (`.browser_id`) |
| Stop from a handler | `self.request_shutdown()` |
| Browser stop key | `VisualizerApp(enable_server_stop_key=True)` (Ctrl+Q) |
| Split-view app | override `run()` to `show(layout=...)` |

**Next:** [16 — Controls](../16_controls/).
